# Exercise B3 — Transfer: Same Method, New Equations

**Optional capstone for Lectures 2 and B.** The claim of the notes is *learn the steps, not the
symbols*. Here you change the equation twice — and watch which parts of the machinery notice.

| | equation | what is new | physical picture |
|---|---|---|---|
| **I** | $\theta'' - m^2\theta = 0$ | one extra **term** | cooling fin (or: bar in elastic soil) |
| **II** | $w'''' = q$ | higher **order** | bending of a beam |
| **III** (bonus) | $-\psi'' = \mu\,\psi$ | unknown **eigenvalue** | particle in a box / buckling / vibration |

Scorekeeping question for the whole notebook: *each time the equation changes, how many lines
change in the FEM route, and how many in the PINN route?*

---
*Author: Ramses Sala · Copyright © 2025 Ramses Sala · Part of the course* Essential Concepts in FEM and Physics-Based ML. *Created with assistance of Claude Code for text, image and code refinement and generation.*

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp

# ---------- helpers (same as in Exercises A and B1) ----------
# num_grad is a transparent stand-in for automatic differentiation: real frameworks
# compute the same numbers exactly and much faster.
def num_grad(f, th, h=1e-6):
    """Numerical gradient of scalar f at parameter vector th (central differences)."""
    g = np.zeros_like(th)
    for i in range(len(th)):
        tp = th.copy(); tm = th.copy(); tp[i] += h; tm[i] -= h
        g[i] = (f(tp) - f(tm)) / (2*h)
    return g

def adam_minimize(f, th0, steps=2000, lr=0.02, verbose_every=None):
    """Adam: gradient descent with per-parameter step adaptation."""
    th = np.asarray(th0, float).copy()
    m = np.zeros_like(th); v = np.zeros_like(th)
    for k in range(steps):
        g = num_grad(f, th)
        m = 0.9*m + 0.1*g
        v = 0.999*v + 0.001*g*g
        mh = m/(1-0.9**(k+1)); vh = v/(1-0.999**(k+1))
        th -= lr*mh/(np.sqrt(vh)+1e-8)
        if verbose_every and k % verbose_every == 0:
            print(f"step {k:5d}   J = {f(th):.4e}")
    return th


In [ ]:
# ---------- tanh network with exact derivatives up to order 4 (given) ----------
# g(x) = sum_j c_j tanh(w_j x + b_j) + c0,   th = [c (H), w (H), b (H), c0]
def net4(x, th, H):
    c = th[:H]; w = th[H:2*H]; b = th[2*H:3*H]; c0 = th[3*H]
    t = np.tanh(np.outer(np.atleast_1d(x), w) + b); s = 1 - t**2
    g  = t @ c + c0
    g1 = (s*w) @ c
    g2 = (-2*t*s*w**2) @ c
    g3 = (-2*w**3*s*(1 - 3*t**2)) @ c
    g4 = (8*w**4*t*s*(2 - 3*t**2)) @ c
    return g, g1, g2, g3, g4

def init_net(H, seed=0):
    rng = np.random.default_rng(seed)
    return np.concatenate([rng.normal(0, 0.1, H), rng.normal(0, 1, H),
                           rng.normal(0, 1, H), [0.0]])


---
## Part 1 — The cooling fin: one extra term

A thin fin (the rod of Exercise 2) now also loses heat through its lateral surface to air at $T_\infty$.
Slice balance: conduction in − conduction out − **convective loss** $h_c P\,(T - T_\infty)\,\Delta x$ = 0
($h_c$: convection coefficient, $P$: perimeter). Fin length $L = 1$ (nondimensional units throughout).
With $\theta = T - T_\infty$ and $m^2 = h_c P/(\lambda A)$:
$$\theta'' - m^2\theta = 0, \qquad \theta(0) = 1 \ (\text{hot base}), \qquad \theta'(1) = 0 \ (\text{insulated tip}).$$

1. **On paper**: redo the slice balance of Exercise 2 with the loss term and derive the equation.
   Which of the three relations (gradient, Fourier, balance) changed?
2. Solve exactly with `sympy` for symbolic $m$. (Expect hyperbolic functions — the solution is no
   longer a polynomial, so no polynomial ansatz will 'contain the truth' any more.)

In [ ]:
x, m = sp.symbols('x m', positive=True)
th = sp.Function('theta')
sol = ...     # TODO: sp.dsolve with ics theta(0)=1, theta'(1)=0
theta_exact = sp.simplify(sol.rhs)
theta_exact

In [ ]:
# ✅ check — Part 1
assert sp.simplify(theta_exact - sp.cosh(m*(1 - x))/sp.cosh(m)).rewrite(sp.exp).simplify() == 0, \
    "Expect theta = cosh(m(1-x)) / cosh(m)."
m_val = 2.0
exact = lambda xv: np.cosh(m_val*(1 - xv))/np.cosh(m_val)
print("OK. From here on m = 2. Tip temperature:", exact(1.0))

---
## Part 2 — FEM by the recipe: a new term needs a new element matrix

Weak form (derive it: one integration by parts, exactly as in Lecture 2):
$\int v'\theta'\,dx + m^2\!\int v\,\theta\,dx = 0$ (weight function $v$). The first integral gives the familiar
$\tfrac1h\begin{bmatrix}1&-1\\-1&1\end{bmatrix}$; the second is new.

1. With `sympy`, derive the **reaction matrix** $\mathbf{m}^e = m^2\int_0^h \mathbf N^T\mathbf N\,d\xi$ for
   linear shape functions.
2. Assemble $\mathbf K = \sum(\mathbf k^e + \mathbf m^e)$ for `ne` elements, impose $\theta_1 = 1$ by
   partitioning, solve.
3. Is the FEM solution still exact at the nodes (as in Lecture 2)? Compute the nodal error for
   2, 4, 8 elements. By what factor does it shrink when $h$ halves?

In [ ]:
xi, h = sp.symbols('xi h', positive=True)
N = sp.Matrix([[1 - xi/h, xi/h]])
me_sym = ...       # TODO: integral of N^T N over the element (without the factor m^2)
print(me_sym)

def fin_fem(ne, m_val=2.0):
    n = ne + 1; hh = 1.0/ne
    K = np.zeros((n, n))
    for e in range(ne):
        ke = ...   # TODO: conduction part
        me = ...   # TODO: reaction part (include m_val**2)
        K[e:e+2, e:e+2] += ke + me
    T = np.zeros(n); T[0] = 1.0
    T[1:] = ...    # TODO: partitioned solve (the prescribed value moves to the right-hand side)
    return np.linspace(0, 1, n), T

errs = []
for ne in [2, 4, 8]:
    xn, T = fin_fem(ne)
    errs.append(np.max(np.abs(T - exact(xn))))
    print(f"{ne} elements: max nodal error = {errs[-1]:.5f}")
print("error ratios:", errs[0]/errs[1], errs[1]/errs[2])

In [ ]:
# ✅ check — Part 2
assert sp.simplify(me_sym - (h/6)*sp.Matrix([[2, 1], [1, 2]])) == sp.zeros(2, 2), "Expect h/6 * [[2,1],[1,2]]."
assert abs(errs[0] - 0.02308) < 1e-3, "2 elements: nodal error about 0.023 (no longer exact!)."
assert 3.5 < errs[1]/errs[2] < 4.5, "Halving h should cut the error by about 4."
print("OK. Nodal exactness was a gift of the old equation, not a law of FEM. Convergence: error ~ h^2.")
print("FEM scorecard for the new term: one new element matrix, derived by the usual recipe.")

---
## Part 3 — PINN forward: change one line

Network ansatz with the essential condition built in: $\tilde\theta(x) = 1 + x\,g(x)$.
Loss: mean squared residual of the *new* equation at 50 collocation points + penalty $(\tilde\theta'(1))^2$.
Compare with your Exercise B1 code: what had to change?

In [ ]:
# given: hard-BC ansatz for the fin, with derivatives
H = 6
x_col = np.linspace(0.01, 1.0, 50)
def theta_hard(xv, th):
    xv = np.atleast_1d(xv).astype(float)
    g, g1, g2, _, _ = net4(xv, th, H)
    return 1 + xv*g, g + xv*g1, 2*g1 + xv*g2

In [ ]:
def fin_loss(th):
    u, du, ddu = theta_hard(x_col, th)
    R = ...                         # TODO: residual of  theta'' - m^2 theta = 0   (m_val = 2)
    _, d1, _ = theta_hard(1.0, th)
    return ...                      # TODO: mean R^2 + tip penalty

th_fin = adam_minimize(fin_loss, init_net(H), steps=3000, lr=0.02, verbose_every=1000)
xx = np.linspace(0, 1, 300)
fin_err = np.max(np.abs(theta_hard(xx, th_fin)[0] - exact(xx)))
print(f"max error PINN = {fin_err:.2e}")
xn, T8 = fin_fem(8)
plt.plot(xx, exact(xx), 'b', label='exact'); plt.plot(xx, theta_hard(xx, th_fin)[0], 'r--', label='PINN')
plt.plot(xn, T8, 'ks', ms=4, label='FEM, 8 elements'); plt.legend(); plt.show()

In [ ]:
# ✅ check — Part 3
assert fin_err < 2e-3
print("OK. PINN scorecard for the new term: one line (the residual). Cost: 3000 steps instead of one solve.")

---
## Part 4 — Inverse: measure temperatures, identify the heat-transfer coefficient

In practice $m^2 = h_c P/(\lambda A)$ is *the* uncertain quantity of fin design (the convection
coefficient $h_c$ is notoriously hard to predict). Five thermocouples give $\theta$ at
$x = 0.2, 0.4, \dots, 1.0$. Train the network **and** $m^2$ jointly:
$$J = \text{mean}\,(\tilde\theta'' - m^2\tilde\theta)^2 + (\tilde\theta'(1))^2 + 20\cdot\text{mean}\,(\tilde\theta(x_j) - \theta_j^{\rm meas})^2.$$
Then repeat with noisy data (σ = 0.005) and see how the estimate moves.

In [ ]:
x_meas = np.linspace(0.2, 1.0, 5)
t_meas = exact(x_meas)                      # synthetic, noise-free (true m^2 = 4)

def fin_inverse_loss(th, data):
    m2 = th[-1]; thn = th[:-1]
    u, du, ddu = theta_hard(x_col, thn)
    _, d1, _ = theta_hard(1.0, thn)
    um = theta_hard(x_meas, thn)[0]
    return ...                              # TODO: physics + tip penalty + 20 * data misfit

th_inv = adam_minimize(lambda th: fin_inverse_loss(th, t_meas),
                       np.concatenate([init_net(H), [1.0]]), steps=3000, lr=0.02, verbose_every=1000)
m2_id = th_inv[-1]
rng = np.random.default_rng(5)
t_noisy = t_meas + 0.005*rng.standard_normal(5)
th_inv_n = adam_minimize(lambda th: fin_inverse_loss(th, t_noisy),
                         np.concatenate([init_net(H), [1.0]]), steps=3000, lr=0.02)
m2_noisy = th_inv_n[-1]
print(f"identified m^2 = {m2_id:.4f} (noise-free)   {m2_noisy:.4f} (noisy)    true: 4")

In [ ]:
# ✅ check — Part 4
assert abs(m2_id - 4.0) < 0.1, "Noise-free data should identify m^2 = 4 closely."
assert abs(m2_noisy - 4.0) < 0.5, "Noisy data: still in the neighbourhood."
print("OK. Note: no shortcut here — unlike the bar, the fin is not 'statically determinate':")
print("the heat flow depends on the unknown m^2 everywhere. The joint training is doing real work.")

---
## Part 5 — The beam: a higher-order equation

Cantilever, $EI = L = q = 1$: $\;w'''' = 1$, clamped at $x=0$ ($w = w' = 0$, **two** essential
conditions), free at $x=1$ ($M = w'' = 0$ and $V = w''' = 0$, **two** natural conditions).
Exact: $w = x^2(6 - 4x + x^2)/24$, tip deflection $1/8$.

Two conjugate pairs per end now: (deflection, shear force) and (slope, bending moment) — the
'one of each pair' rule of Lecture 1, doubled.

1. **Pencil Ritz**: with the one-parameter ansatz $\tilde w = a\,x^2$ (satisfies both essential
   conditions), minimize the energy $\Pi(a) = \int_0^1 \tfrac12(\tilde w'')^2\,dx - \int_0^1 \tilde w\,dx$.
   How far off is the tip deflection?
2. **PINN**: ansatz $\tilde w = x^2 g(x)$ (both essential conditions hard — why does $x^2$ do it?).
   Loss: mean $(\tilde w'''' - 1)^2$ + penalties $(\tilde w''(1))^2 + (\tilde w'''(1))^2$.
3. FEM for this equation needs a *new element type* (shape functions with continuous slope —
   beyond these notes). What did the PINN need?

In [ ]:
a = sp.Symbol('a')
w_ritz = a*x**2
Pi = ...                                  # TODO: energy of the one-parameter ansatz
a_ritz = sp.solve(sp.diff(Pi, a), a)[0]
print("Ritz: a =", a_ritz, "  tip =", a_ritz, "  exact tip = 1/8")

def w_hard(xv, th):                       # given: w = x^2 g and its derivatives w'', w''', w''''
    xv = np.atleast_1d(xv).astype(float)
    g, g1, g2, g3, g4 = net4(xv, th, H)
    return xv**2*g, 2*g + 4*xv*g1 + xv**2*g2, 6*g1 + 6*xv*g2 + xv**2*g3, 12*g2 + 8*xv*g3 + xv**2*g4

def beam_loss(th):
    w, w2, w3, w4 = w_hard(x_col, th)
    _, M1, V1, _ = w_hard(1.0, th)
    return ...                            # TODO: residual + two natural-BC penalties

th_beam = adam_minimize(beam_loss, init_net(H), steps=3000, lr=0.02, verbose_every=1000)
tip = w_hard(1.0, th_beam)[0][0]
print(f"PINN tip deflection = {tip:.5f}   (exact 0.125)")

In [ ]:
# ✅ check — Part 5
assert a_ritz == sp.Rational(1, 12), "One-parameter Ritz: a = 1/12 (tip 0.0833, a third too small)."
assert abs(tip - 0.125) < 1e-3
print("OK. PINN scorecard for a 4th-order equation: a different residual line and two more penalties.")
print("FEM scorecard: a new element family. (But once built, it solves every beam in one linear solve.)")

---
## Part 6 (bonus) — An eigenvalue problem, and a trap

$-\psi'' = \mu\,\psi$ on $(0,1)$, $\psi(0) = \psi(1) = 0$: the stationary Schrödinger equation of a
particle in a box — and, with other letters, the buckling of a column or the vibration of a string.
The number $\mu$ is **unknown**, like the material parameter of the inverse problems (physicists
write $E$, the energy; here $\mu$, since $E$ is Young's modulus in the notes); the smallest
solution is $\mu = \pi^2 \approx 9.87$ with $\psi = \sin\pi x$.

**The trap.** Run the given residual-PINN below (it trains the network and $\mu$ together, with a
penalty asking for $\int\psi^2 = 1$). Look at what it converges to — and explain.

In [ ]:
# given: residual PINN for the eigenvalue problem. Run it and look closely.
x_q = np.linspace(0, 1, 101); w_q = np.ones(101)/100; w_q[0] = w_q[-1] = 0.5/100
def psi_hard(xv, th):                     # psi = x(1-x) g : both boundary conditions hard
    g, g1, g2, _, _ = net4(xv, th, H)
    p = xv*(1 - xv); p1 = 1 - 2*xv
    return p*g, p1*g + p*g1, -2*g + 2*p1*g1 + p*g2
def eig_residual_loss(th):
    mu = th[-1]; ps, dps, ddps = psi_hard(x_q, th[:-1])
    return np.mean((ddps + mu*ps)**2) + (np.sum(w_q*ps**2) - 1)**2
th_trap = adam_minimize(eig_residual_loss, np.concatenate([init_net(H), [5.0]]), steps=1500, lr=0.02)
ps_trap = psi_hard(x_q, th_trap[:-1])[0]
print(f"mu = {th_trap[-1]:.3f}    integral of psi^2 = {np.sum(w_q*ps_trap**2):.2e}    loss = {eig_residual_loss(th_trap):.3f}")

**The way out is the energy form.** The eigenvalue is the minimum of the *Rayleigh quotient*
$$\mu[\psi] = \frac{\int_0^1 (\psi')^2\,dx}{\int_0^1 \psi^2\,dx},$$
which is blind to the scale of $\psi$ — the trivial solution is not a minimum, it is not even in
the domain.

1. **Pencil**: evaluate the quotient for the crudest admissible function, $\psi = x(1-x)$.
   How close to $\pi^2$?
2. Minimize the quotient over the network $\psi = x(1-x)\,g(x)$.

In [ ]:
psi_p = x*(1 - x)
mu_pencil = ...                            # TODO: Rayleigh quotient of x(1-x), exactly (sympy)
print("pencil estimate:", mu_pencil, "   pi^2 =", float(sp.pi**2))

def rayleigh(th):
    ps, dps, _ = psi_hard(x_q, th)
    return ...                            # TODO: quotient of the two quadrature sums (weights w_q)

th0 = init_net(H); th0[-1] = 1.0
th_eig = adam_minimize(rayleigh, th0, steps=1500, lr=0.02)
mu_net = rayleigh(th_eig)
print(f"network estimate: mu = {mu_net:.5f}")

In [ ]:
# ✅ check — Part 6
assert mu_pencil == 10, "The pencil estimate is exactly 10 — within 1.3 % of pi^2."
assert abs(mu_net - np.pi**2) < 1e-2
print("OK. A residual can be zero for the wrong reason (psi = 0): small residual, total error.")
print("The energy form has no such trap — and its minimum VALUE is the answer.")

---
## Closing: the scorecard

Fill in (markdown) for each of the three transfers — what changed, in lines of code or in concepts?

| | FEM route | PINN route |
|---|---|---|
| extra term (fin) | | |
| higher order (beam) | | |
| unknown eigenvalue | | |

Then answer: *the PINN route changed less every time. Is that an argument for PINNs?*
(Consider what each route gave you in return — accuracy per unit of work, guarantees, reuse.)